In [1]:
import requests
import pandas as pd
import os
import time
from dotenv import load_dotenv

In [ ]:
FRED_API_KEY = "you api key" #get free FRED api key and put it here
FRED_BASE = "https://api.stlouisfed.org/fred/series/observations"
print(f"Key: '{FRED_API_KEY}'")


Key: '313e129a45dfd1a465ea13dac85decf7'


In [3]:
FRED_SERIES = {
    "FEDFUNDS": "Federal Funds Rate",
    "MORTGAGE30US": "30-Year Fixed Mortgage Rate",
    "DGS10": "10-Year Treasury Rate",
    "T10Y2Y": "10Y-2Y Yield Spread",
    "UNRATE": "Unemployment Rate",
    "CPIAUCSL": "Consumer Price Index",
    "DRALACBS": "Delinquency Rate All Loans",
    "DRSFRMACBS": "Delinquency Rate Mortgage SFR",
    "GDP": "Gross Domestic Product",
    "CSUSHPINSA": "Case-Shiller Home Price Index",
}

def scrape_fred_series(
    series_id: str,
    api_key: str,
    start_date: str = "2019-01-01",
) -> pd.DataFrame:
    """Scrape 1 FRED series through REST API."""
    params = {
        "series_id": series_id,
        "api_key": api_key,
        "file_type": "json",
        "observation_start": start_date,
    }
    
    response = requests.get(FRED_BASE, params=params)
    response.raise_for_status()

    observations = response.json()["observations"]
    df = pd.DataFrame(observations)[["date", "value"]]
    df.columns = ["date", series_id.lower()]
    df["date"] = pd.to_datetime(df["date"])
    df[series_id.lower()] = pd.to_numeric(df[series_id.lower()], errors="coerce")
    return df

In [4]:
#scrape all series
print("=" * 60)
print("SCRAPING FRED MACRO DATA")
print("=" * 60)
macro_frames = {}
for sid, name in FRED_SERIES.items():
    print(f"[SCRAPING] {sid} — {name}")
    df_s = scrape_fred_series(sid, FRED_API_KEY)
    macro_frames[sid] = df_s
    print(f"  → {len(df_s)} observations ({df_s['date'].min()} to {df_s['date'].max()})")
    time.sleep(0.3)

SCRAPING FRED MACRO DATA
[SCRAPING] FEDFUNDS — Federal Funds Rate
  → 91 observations (2019-01-01 00:00:00 to 2026-07-01 00:00:00)
[SCRAPING] MORTGAGE30US — 30-Year Fixed Mortgage Rate
  → 400 observations (2019-01-03 00:00:00 to 2026-08-27 00:00:00)
[SCRAPING] DGS10 — 10-Year Treasury Rate
  → 1998 observations (2019-01-01 00:00:00 to 2026-08-27 00:00:00)
[SCRAPING] T10Y2Y — 10Y-2Y Yield Spread
  → 1999 observations (2019-01-01 00:00:00 to 2026-08-28 00:00:00)
[SCRAPING] UNRATE — Unemployment Rate
  → 91 observations (2019-01-01 00:00:00 to 2026-07-01 00:00:00)
[SCRAPING] CPIAUCSL — Consumer Price Index
  → 91 observations (2019-01-01 00:00:00 to 2026-07-01 00:00:00)
[SCRAPING] DRALACBS — Delinquency Rate All Loans
  → 30 observations (2019-01-01 00:00:00 to 2026-04-01 00:00:00)
[SCRAPING] DRSFRMACBS — Delinquency Rate Mortgage SFR
  → 30 observations (2019-01-01 00:00:00 to 2026-04-01 00:00:00)
[SCRAPING] GDP — Gross Domestic Product
  → 30 observations (2019-01-01 00:00:00 to 2026-0

In [5]:
#Merge all series into a single DataFrame
df_macro = macro_frames[list(FRED_SERIES.keys())[0]]
for sid in list(FRED_SERIES.keys())[1:]:
    df_macro = pd.merge(df_macro, macro_frames[sid], on="date", how="outer")
df_macro = df_macro.sort_values("date").reset_index(drop=True)
print(f"\nMerged macro data: {df_macro.shape}")
print(f"Date range: {df_macro['date'].min()} → {df_macro['date'].max()}")


Merged macro data: (2024, 11)
Date range: 2019-01-01 00:00:00 → 2026-08-28 00:00:00


In [ ]:
os.makedirs("../A. Data Pipeline/Data/bronze/", exist_ok=True)
df_macro.to_csv("../A. Data Pipeline/Data/bronze/fred_macro_raw.csv", index=False)
print("Bronze: fred_macro_raw.csv saved.")

Bronze: fred_macro_raw.csv saved.
